# 06 — Red con balance detallado y diagnóstico del cuello de botella del deuterio

Este notebook crea una segunda red `pynucastro_db`, con tasas inversas añadidas por balance detallado siempre que la instalación local de `pynucastro` lo permita. El objetivo no es esconder la discrepancia de D/H, sino probar la hipótesis física que salió de la comparación con AlterBBN:

1. `pynucastro` reproduce razonablemente $Y_p$, porque casi todos los neutrones iniciales acaban en $^4$He.
2. Sin embargo, D/H sale demasiado bajo.
3. Como $^3$He/H no está tan lejos en escala, el problema no parece ser que nunca se forme deuterio, sino que se quema demasiado eficientemente una vez formado.
4. La sospecha natural es que el cuello de botella del deuterio está incompleto: faltan o no pesan suficientemente las reacciones inversas/fotodesintegraciones, en especial $d + \gamma ightarrow n+p$.

El script `06_make_detailed_balance_network.py` genera la red, exporta un diagnóstico de canales inversos y, si la integración funciona, añade `pynucastro_db` a las tablas y figuras comparativas.

In [ ]:
from pathlib import Path
import subprocess
import sys

script = Path('06_make_detailed_balance_network.py')
if not script.exists():
    raise FileNotFoundError(script)

subprocess.run([sys.executable, str(script)], check=True)

In [ ]:
import pandas as pd
from pathlib import Path

def show_csv(path):
    path = Path(path)
    print(f"\n--- {path} ---")
    if path.exists():
        display(pd.read_csv(path))
    else:
        print("No existe todavía")

show_csv('data/pynucastro_db_status.csv')
show_csv('data/bbn_network_reverse_diagnostic.csv')
show_csv('data/bbn_network_db_derived_attempts.csv')
show_csv('data/bbn_final_abundances_pynucastro_db.csv')

## Qué mirar

La prueba decisiva es la fila `d_to_n_p` en `bbn_network_reverse_diagnostic.csv`. Si aparece como `found = True`, la red `pynucastro_db` sí contiene una vía inversa directa para la reacción de formación del deuterio. Después hay que mirar el valor final de D/H en `pynucastro_db`:

- Si D/H sube hacia $10^{-5}$, la discrepancia principal venía de no tratar bien el equilibrio directo-inverso del deuterio.
- Si D/H sigue muy bajo, entonces la limitación no es solo la red nuclear: también pesa la trayectoria termodinámica impuesta, la ausencia de evolución cosmológica completa, las tasas débiles completas y la física de fotones, electrones, positrones y neutrinos.

El notebook `05_alterbbn_benchmark.ipynb` detecta automáticamente la fila `pynucastro_db` si existe en `bbn_final_abundances_all_models.csv`. Por tanto, después de este notebook conviene ejecutar de nuevo el benchmark para regenerar las figuras finales.